<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter6/WordPiece_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WordPiece tokenization

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


## Implementing WordPiece ##

In [2]:
corpus = [
    "This is the Hugging Face Course.",
    "I love Hugging Face.",
    "My name is Alex and I am learning the Hugging face API.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [5]:
from collections import defaultdict

word_freqs = defaultdict(int)
for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

word_freqs

defaultdict(int,
            {'This': 1,
             'is': 2,
             'the': 2,
             'Hugging': 3,
             'Face': 2,
             'Course': 1,
             '.': 4,
             'I': 2,
             'love': 1,
             'My': 1,
             'name': 1,
             'Alex': 1,
             'and': 2,
             'am': 1,
             'learning': 1,
             'face': 1,
             'API': 1,
             'Hopefully': 1,
             ',': 1,
             'you': 1,
             'will': 1,
             'be': 1,
             'able': 1,
             'to': 1,
             'understand': 1,
             'how': 1,
             'they': 1,
             'are': 1,
             'trained': 1,
             'generate': 1,
             'tokens': 1})

As we saw before, the alphabet is the unique set composed of all the first letters of words, and all the other letters that appear in words prefixed by ##:

In [6]:
alphabet = []
for word in word_freqs.keys():
  if word[0] not in alphabet:
    alphabet.append(word[0])
  for letter in word[1:]:
    if f"##{letter}" not in alphabet:
      alphabet.append(f"##{letter}")

alphabet.sort()
alphabet

['##I',
 '##P',
 '##a',
 '##b',
 '##c',
 '##d',
 '##e',
 '##f',
 '##g',
 '##h',
 '##i',
 '##k',
 '##l',
 '##m',
 '##n',
 '##o',
 '##p',
 '##r',
 '##s',
 '##t',
 '##u',
 '##v',
 '##w',
 '##x',
 '##y',
 ',',
 '.',
 'A',
 'C',
 'F',
 'H',
 'I',
 'M',
 'T',
 'a',
 'b',
 'f',
 'g',
 'h',
 'i',
 'l',
 'n',
 't',
 'u',
 'w',
 'y']

We also add the special tokens used by the model at the beginning of that vocabulary.

In the case of BERT, it's the list ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]:

In [9]:
vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + alphabet.copy()

In [11]:
splits = {
    word: [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
    for word in word_freqs.keys()
}
splits

{'This': ['T', '##h', '##i', '##s'],
 'is': ['i', '##s'],
 'the': ['t', '##h', '##e'],
 'Hugging': ['H', '##u', '##g', '##g', '##i', '##n', '##g'],
 'Face': ['F', '##a', '##c', '##e'],
 'Course': ['C', '##o', '##u', '##r', '##s', '##e'],
 '.': ['.'],
 'I': ['I'],
 'love': ['l', '##o', '##v', '##e'],
 'My': ['M', '##y'],
 'name': ['n', '##a', '##m', '##e'],
 'Alex': ['A', '##l', '##e', '##x'],
 'and': ['a', '##n', '##d'],
 'am': ['a', '##m'],
 'learning': ['l', '##e', '##a', '##r', '##n', '##i', '##n', '##g'],
 'face': ['f', '##a', '##c', '##e'],
 'API': ['A', '##P', '##I'],
 'Hopefully': ['H', '##o', '##p', '##e', '##f', '##u', '##l', '##l', '##y'],
 ',': [','],
 'you': ['y', '##o', '##u'],
 'will': ['w', '##i', '##l', '##l'],
 'be': ['b', '##e'],
 'able': ['a', '##b', '##l', '##e'],
 'to': ['t', '##o'],
 'understand': ['u',
  '##n',
  '##d',
  '##e',
  '##r',
  '##s',
  '##t',
  '##a',
  '##n',
  '##d'],
 'how': ['h', '##o', '##w'],
 'they': ['t', '##h', '##e', '##y'],
 'are': ['a', '

Now that we are ready for training, let's write a function that computes the score of each pair. We'll need to use this at each step of the training:

In [12]:
def compute_pair_scores(splits):
  letter_freqs = defaultdict(int)
  pair_freqs = defaultdict(int)
  for word, freq in word_freqs.items():
    split = splits[word]
    if len(split) == 1:
      letter_freqs[split[0]] += freq
      continue
    for i in range(len(split) - 1):
      pair = (split[i], split[i+1])
      letter_freqs[split[i]] += freq
      pair_freqs[pair] += freq
    letter_freqs[split[-1]] += freq

  scores = {
      pair: freq / (letter_freqs[pair[0]] * letter_freqs[pair[1]])
      for pair, freq in pair_freqs.items()
  }
  return scores

Let’s have a look at a part of this dictionary after the initial splits:

In [13]:
pair_scores = compute_pair_scores(splits)
for i, key in enumerate(pair_scores.keys()):
  print(f"{key}: {pair_scores[key]}")
  if i >= 5:
    break

('T', '##h'): 0.25
('##h', '##i'): 0.03571428571428571
('##i', '##s'): 0.023809523809523808
('i', '##s'): 0.16666666666666666
('t', '##h'): 0.125
('##h', '##e'): 0.03571428571428571


In [14]:
best_pair = ""
max_score = None
for pair, score in pair_scores.items():
  if max_score is None or max_score < score:
    best_pair = pair
    max_score = score

best_pair, max_score

(('##P', '##I'), 1.0)

So the first merge to learn is ('##P', '##I') -> PI, and we add 'PI' to the vocabulary:

In [15]:
vocab.append("PI")

To continue, we need to apply that merge in our splits dictionary. Let’s write another function for this:

In [20]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue
        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                merge = a + b[2:] if b.startswith("##") else a + b
                split = split[:i] + [merge] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

In [22]:
splits = merge_pair('##P', '##I', splits)
splits['API']

['A', '##PI']

Now we have everything we need to loop until we have learned all the merges we want. Let's aim for a vocab size of 70:

In [24]:
vocab_size = 70
while len(vocab) < vocab_size:
  scores = compute_pair_scores(splits)
  best_pair, max_score = "", None
  for pair, score in scores.items():
    if max_score is None or max_score < score:
      best_pair = pair
      max_score = score
  splits = merge_pair(*best_pair, splits)
  new_token = (
      best_pair[0] + best_pair[1][2:]
      if best_pair[1].startswith("##")
      else best_pair[0] + best_pair[1]
  )
  vocab.append(new_token)
vocab

['[PAD]',
 '[UNK]',
 '[CLS]',
 '[SEP]',
 '[MASK]',
 '##I',
 '##P',
 '##a',
 '##b',
 '##c',
 '##d',
 '##e',
 '##f',
 '##g',
 '##h',
 '##i',
 '##k',
 '##l',
 '##m',
 '##n',
 '##o',
 '##p',
 '##r',
 '##s',
 '##t',
 '##u',
 '##v',
 '##w',
 '##x',
 '##y',
 ',',
 '.',
 'A',
 'C',
 'F',
 'H',
 'I',
 'M',
 'T',
 'a',
 'b',
 'f',
 'g',
 'h',
 'i',
 'l',
 'n',
 't',
 'u',
 'w',
 'y',
 'PI',
 'My',
 'Th',
 'ab',
 'is',
 'th',
 'Al',
 'abl',
 '##fu',
 '##ful',
 '##full',
 '##fully',
 '##ll',
 'Hu',
 'Thi',
 'This',
 'wi',
 'will',
 '##st']

To tokenize a new text, we pre-tokenize it, split it, then apply the tokenization algorithm on each word.

That is, we look for the biggest subword starting at the beginning of the first word and split it, then we repeat the process on the second part, and so on for the rest of that word and the following words in the text:

In [25]:
def encode_word(word):
  tokens = []
  while len(word) > 0:
    i = len(word)
    while i > 0 and word[:i] not in vocab:
      i -= 1
    if i == 0:
      return ["[UNK]"]
    tokens.append(word[:i])
    word = word[i:]
    if len(word) > 0:
      word = f"##{word}"
  return tokens

Now let's test it on one word that's in the vocabulary, and another that isn't:

In [26]:
encode_word("Hugging"), encode_word("H0gging")

(['Hu', '##g', '##g', '##i', '##n', '##g'], ['[UNK]'])

Now, let's write a function that tokenizes a text:

In [34]:
def tokenize(text):
  pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)

  pre_tokenized_text = [word for word, offset in pre_tokenize_result]
  encoded_words = [encode_word(word) for word in pre_tokenized_text]
  print(f"{pre_tokenize_result}\n\n{pre_tokenized_text}\n\n{encoded_words}\n\n")
  return sum(encoded_words, [])

In [35]:
tokenize("I love Hugging Face!")

[('I', (0, 1)), ('love', (2, 6)), ('Hugging', (7, 14)), ('Face', (15, 19)), ('!', (19, 20))]

['I', 'love', 'Hugging', 'Face', '!']

[['I'], ['l', '##o', '##v', '##e'], ['Hu', '##g', '##g', '##i', '##n', '##g'], ['F', '##a', '##c', '##e'], ['[UNK]']]




['I',
 'l',
 '##o',
 '##v',
 '##e',
 'Hu',
 '##g',
 '##g',
 '##i',
 '##n',
 '##g',
 'F',
 '##a',
 '##c',
 '##e',
 '[UNK]']